# End-to-End: Preprocessing & Training Multi-Task BiLSTM untuk Estimasi Tekanan Darah

**Topik Skripsi**: Estimasi Tekanan Darah Non-Invasif Berbasis Sinyal ECG Menggunakan Deep Learning  
**Dataset**: *Cuff-Less Blood Pressure Estimation Dataset* (Kachuee et al., 2016 / PhysioNet MIMIC-II)  
**Arsitektur**: Multi-Task Bidirectional LSTM (Fan et al., 2021, Sensors)  
**Akselerator**: GPU T4 x2 (Kaggle)

---

### Daftar Isi
| Fase | Blok | Deskripsi |
|---|---|---|
| **I. Preprocessing** | 1–2 | Setup, fungsi preprocessing inline, load dataset |
| | 3 | Ringkasan statistik & visualisasi distribusi Ground Truth BP |
| | 4 | Visualisasi sinyal multi-channel per part |
| | 5 | Demonstrasi preprocessing step-by-step (1 epoch contoh) |
| | 6 | Batch preprocessing seluruh dataset → simpan `.npz` |
| **II. Training** | 7 | Definisi model BiLSTM + Adaptive Loss |
| | 8 | Dataset PyTorch & DataLoader (split 80/10/10) |
| | 9 | Training loop (Fan et al., 2021: Momentum-SGD, 500 epoch) |
| | 10 | Visualisasi kurva training |
| **III. Evaluasi** | 11 | Evaluasi akhir test set (BHS & AAMI) |
| | 12 | Scatter plot prediksi vs ground truth |
| | 13 | Bland-Altman agreement analysis |
| | 14 | Simpan semua artefak |

---

### Petunjuk Kaggle
1. **Add Data** → cari `mkachuee/BloodPressureDataset` → tambahkan
2. **Settings** → Accelerator → pilih **GPU T4 x2**
3. **Run All** — notebook ini memproses seluruh dataset, melatih, dan mengevaluasi model

---
# FASE I — PREPROCESSING

---
## 1. Setup & Import

In [ ]:
import os
import glob
import gc
import time
import json
import numpy as np
import pandas as pd
import scipy.io
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import iirnotch, butter, filtfilt, find_peaks
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# ── Deteksi akselerator ──────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f'✅ GPU terdeteksi: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('✅ Apple Silicon MPS terdeteksi')
else:
    DEVICE = torch.device('cpu')
    print('⚠️ Tidak ada GPU — menggunakan CPU (akan sangat lambat!)')

print(f'PyTorch: {torch.__version__} | Device: {DEVICE}')

---
## 2. Hyperparameter & Fungsi Preprocessing (Inline)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                       HYPERPARAMETER GLOBAL
# ══════════════════════════════════════════════════════════════════════════════

# -- Preprocessing --
FS = 125.0                          # Sampling rate dataset Kachuee (Hz)
EPOCH_SEC = 10.0                    # Durasi 1 epoch (detik)
EPOCH_LEN = int(EPOCH_SEC * FS)     # = 1250 sampel per epoch

NOTCH_FREQ = 50.0                   # Frekuensi notch PLN (Hz)
NOTCH_Q = 30.0                      # Quality factor notch filter
BP_LOWCUT = 0.5                     # Bandpass low cutoff (Hz)
BP_HIGHCUT = 40.0                   # Bandpass high cutoff (Hz)
BP_ORDER = 2                        # Orde Butterworth (efektif orde 4 via filtfilt)

# -- Filter kualitas BP --
SBP_RANGE = (50.0, 220.0)           # mmHg
DBP_RANGE = (30.0, 140.0)           # mmHg

# -- Model BiLSTM (Tabel 1 Fan et al., 2021) --
INPUT_SIZE = 1
HIDDEN_SIZE = 256                   # BiLSTM hidden (output = 2×256 = 512)
NUM_LAYERS = 2
LSTM_DROPOUT = 0.3                  # Dropout antar layer LSTM
FC_DROPOUT = 0.2                    # Dropout di FC heads

# -- Training (Fan et al., 2021) --
BATCH_SIZE = 128
NUM_EPOCHS = 500                    # Sesuai Fan et al. (2021): 500 epoch (ubah ke 30-50 jika ingin quick run di Kaggle)
LEARNING_RATE = 8e-4                # Sesuai Fan et al. (2021): 0.0008
OPTIMIZER_TYPE = 'SGD'              # 'SGD' (Fan et al., 2021: Momentum-SGD) atau 'AdamW' (alternatif modern)
MOMENTUM = 0.9                      # Sesuai Fan et al. (2021): 0.9
WEIGHT_DECAY = 0.1                  # Sesuai Fan et al. (2021): L2 penalty = 0.1
GRAD_CLIP = 5.0
SCHEDULER_FACTOR = 0.9              # Sesuai Fan et al. (2021): ReduceLROnPlateau factor 0.9
SCHEDULER_PATIENCE = 4              # Sesuai Fan et al. (2021): ReduceLROnPlateau patience 4

# -- Data Split --
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10
SEED = 42

# -- Output --
OUTPUT_DIR = '/kaggle/working'
PROCESSED_DIR = os.path.join(OUTPUT_DIR, 'processed')
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Hyperparameter siap.')
print(f'  Seq Length  : {EPOCH_LEN} sampel ({EPOCH_SEC}s @ {FS} Hz)')
print(f'  Preprocessing : Notch {NOTCH_FREQ}Hz (Q={NOTCH_Q}) → BPF {BP_LOWCUT}–{BP_HIGHCUT}Hz → MinMax [0,1]')
print(f'  Model       : BiLSTM {NUM_LAYERS}×{HIDDEN_SIZE} (output={HIDDEN_SIZE*2})')
print(f'  Training    : {NUM_EPOCHS} epoch, batch={BATCH_SIZE}, opt={OPTIMIZER_TYPE}, lr={LEARNING_RATE}')


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                FUNGSI PREPROCESSING (SELF-CONTAINED / INLINE)
# Mereplikasi src/preprocessing/filters.py & src/preprocessing/normalization.py
# ══════════════════════════════════════════════════════════════════════════════

def iir_notch_filter(signal, fs=125.0, f0=50.0, q=30.0):
    """
    IIR Notch Filter — Eliminasi interferensi frekuensi listrik PLN 50 Hz.
    Metode: Zero-Phase Filtering (filtfilt) sesuai Alhamid (2026) & Hakim (2025).
    """
    if signal.size == 0:
        return signal
    b, a = iirnotch(w0=f0, Q=q, fs=fs)
    return filtfilt(b, a, signal, axis=0)


def butterworth_bandpass_filter(signal, fs=125.0, lowcut=0.5, highcut=40.0, order=2):
    """
    Butterworth Bandpass Filter 0.5–40 Hz.
    Zero-Phase Filtering (filtfilt) → orde efektif = 2 × order = 4.
    """
    if signal.size == 0:
        return signal
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    if high >= 1.0:
        high = 0.99
    b, a = butter(N=order, Wn=[low, high], btype='bandpass')
    return filtfilt(b, a, signal, axis=0)


def minmax_scale(signal, eps=1e-8):
    """
    Min-Max Normalization [0, 1] sesuai Fan et al. (2021) & Li et al. (2020).
    Formula: x_norm = (signal - x_min) / (x_max - x_min + eps)
    """
    if signal.size == 0:
        return signal
    x_min = np.min(signal)
    x_max = np.max(signal)
    range_val = x_max - x_min
    if range_val < eps:
        range_val = 1.0
    return (signal - x_min) / range_val


def preprocess_ecg_pipeline(signal, fs=125.0):
    """Pipeline preprocessing lengkap: Notch → Bandpass → MinMax [0,1]."""
    signal = iir_notch_filter(signal, fs=fs, f0=NOTCH_FREQ, q=NOTCH_Q)
    signal = butterworth_bandpass_filter(signal, fs=fs, lowcut=BP_LOWCUT, highcut=BP_HIGHCUT, order=BP_ORDER)
    signal = minmax_scale(signal)
    return signal


def compute_bp_metrics(abp_signal, fs=125.0):
    """
    Menghitung SBP, DBP, MAP dari sinyal ABP menggunakan deteksi puncak per beat.
    Metode beat-by-beat sesuai standar klinis.
    """
    min_peak_distance = int(0.3 * fs)
    min_peak_height = np.percentile(abp_signal, 50)

    systolic_peaks, _ = find_peaks(abp_signal, distance=min_peak_distance, height=min_peak_height)
    diastolic_troughs, _ = find_peaks(-abp_signal, distance=min_peak_distance)

    sbp = float(np.mean(abp_signal[systolic_peaks])) if len(systolic_peaks) > 0 else float(np.max(abp_signal))
    dbp = float(np.mean(abp_signal[diastolic_troughs])) if len(diastolic_troughs) > 0 else float(np.min(abp_signal))
    map_val = float(np.mean(abp_signal))

    is_valid = (
        len(systolic_peaks) > 0
        and SBP_RANGE[0] <= sbp <= SBP_RANGE[1]
        and DBP_RANGE[0] <= dbp <= DBP_RANGE[1]
    )
    return {
        'sbp': sbp, 'dbp': dbp, 'map': map_val,
        'n_beats': len(systolic_peaks), 'valid': is_valid,
        'systolic_peaks_idx': systolic_peaks,
        'diastolic_troughs_idx': diastolic_troughs,
    }


# ── Fungsi Pemuat Dataset ────────────────────────────────────────────────────

def load_mat_file(file_path):
    """Membaca file .mat (MATLAB v5/v7 atau HDF5 v7.3)."""
    try:
        mat = scipy.io.loadmat(file_path)
        keys = [k for k in mat.keys() if not k.startswith('__')]
        return mat[keys[0]]
    except Exception:
        import h5py
        with h5py.File(file_path, 'r') as f:
            return np.array(f[list(f.keys())[0]])


def extract_record_matrix(cell_data, index):
    """Ekstrak matriks sinyal [3, N_samples] dari cell array."""
    if isinstance(cell_data, np.ndarray) and cell_data.dtype == object:
        raw = cell_data[0, index]
    else:
        raw = cell_data[index]
    matrix = np.array(raw, dtype=np.float64)
    if matrix.ndim == 2 and matrix.shape[0] > matrix.shape[1] and matrix.shape[1] == 3:
        matrix = matrix.T
    return matrix


def get_num_records(cell_data):
    """Hitung jumlah rekaman dalam cell array."""
    if isinstance(cell_data, np.ndarray) and cell_data.dtype == object:
        return cell_data.shape[1]
    return cell_data.shape[0]


print('✅ Semua fungsi preprocessing & pemuat dataset siap.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#                         PEMUATAN DATASET
# ══════════════════════════════════════════════════════════════════════════════

KAGGLE_DATA_DIR = '/kaggle/input/bloodpressuredataset'

if not os.path.exists(KAGGLE_DATA_DIR):
    alt_paths = [
        '/kaggle/input/BloodPressureDataset',
        '/kaggle/input/bloodpressuredataset/BloodPressureDataset',
    ]
    for p in alt_paths:
        if os.path.exists(p):
            KAGGLE_DATA_DIR = p
            break
    else:
        try:
            import kagglehub
            KAGGLE_DATA_DIR = kagglehub.dataset_download('mkachuee/BloodPressureDataset')
        except ImportError:
            raise FileNotFoundError(
                'Dataset tidak ditemukan. Di Kaggle: Add Data → mkachuee/BloodPressureDataset'
            )

mat_files = sorted(glob.glob(os.path.join(KAGGLE_DATA_DIR, '**', '*.mat'), recursive=True))
print(f'Direktori dataset: {KAGGLE_DATA_DIR}')
print(f'Ditemukan {len(mat_files)} file .mat:')
for f in mat_files:
    size_mb = os.path.getsize(f) / 1024**2
    print(f'  • {os.path.basename(f)} ({size_mb:.1f} MB)')

---
## 3. Ringkasan Statistik & Distribusi Ground Truth BP

In [ ]:
# Ringkasan Statistik per Part
records_summary = []
loaded_parts = {}

print('Memproses dan mengekstraksi metrik statistik dari setiap part...')
for mat_path in mat_files:
    part_name = os.path.basename(mat_path)
    cell_array = load_mat_file(mat_path)
    if cell_array is None:
        continue
    num_records = get_num_records(cell_array)
    loaded_parts[part_name] = cell_array

    sbp_list, dbp_list, map_list = [], [], []
    sample_step = max(1, num_records // 50)
    for idx in range(0, num_records, sample_step):
        try:
            rec = extract_record_matrix(cell_array, idx)
        except Exception:
            continue
        if rec is not None and rec.ndim == 2 and rec.shape[0] >= 3:
            bp = compute_bp_metrics(rec[1, :], fs=FS)
            if bp['valid']:
                sbp_list.append(bp['sbp'])
                dbp_list.append(bp['dbp'])
                map_list.append(bp['map'])

    records_summary.append({
        'Part': part_name, 'Jumlah Rekaman': num_records,
        'Mean SBP': f"{np.mean(sbp_list):.1f}" if sbp_list else 'N/A',
        'Mean DBP': f"{np.mean(dbp_list):.1f}" if dbp_list else 'N/A',
        'Mean MAP': f"{np.mean(map_list):.1f}" if map_list else 'N/A',
        'Sampel Valid': len(sbp_list)
    })

df_summary = pd.DataFrame(records_summary)
print('\n--- Ringkasan Dataset per File Part ---')
display(df_summary)

In [ ]:
# Agregat Ground Truth BP dari seluruh part
all_sbp, all_dbp, all_map, part_labels = [], [], [], []
for part_name, cell_array in loaded_parts.items():
    num_records = get_num_records(cell_array)
    step = max(1, num_records // 60)
    for idx in range(0, num_records, step):
        try:
            rec = extract_record_matrix(cell_array, idx)
        except Exception:
            continue
        if rec is not None and rec.ndim == 2 and rec.shape[0] >= 3:
            bp = compute_bp_metrics(rec[1, :], fs=FS)
            if bp['valid']:
                all_sbp.append(bp['sbp']); all_dbp.append(bp['dbp'])
                all_map.append(bp['map']); part_labels.append(part_name)

df_all_bp = pd.DataFrame({'Part': part_labels, 'SBP': all_sbp, 'DBP': all_dbp, 'MAP': all_map})
print(f'Total sampel Ground Truth BP tervalidasi: {len(df_all_bp)}')
print(df_all_bp[['SBP', 'DBP', 'MAP']].describe().round(2))

In [ ]:
# Histogram & KDE distribusi SBP, DBP, MAP
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
bp_configs = [
    ('SBP', '#1f77b4', '#0d4b75', [(120, 'green', 'Normal (120)'), (140, 'darkorange', 'Hipertensi (140)')]),
    ('DBP', '#2ca02c', '#1b611b', [(80, 'green', 'Normal (80)'), (90, 'darkorange', 'Hipertensi (90)')]),
    ('MAP', '#ff7f0e', '#b35500', [(70, 'green', 'Perfusi Min (70)'), (105, 'darkorange', 'Perfusi Tinggi (105)')]),
]
for i, (col, hist_c, kde_c, lines) in enumerate(bp_configs):
    axes[i].hist(df_all_bp[col], bins=35, density=True, color=hist_c, alpha=0.65, edgecolor='black')
    df_all_bp[col].plot(kind='kde', ax=axes[i], color=kde_c, lw=2)
    axes[i].axvline(df_all_bp[col].mean(), color='red', linestyle='--', lw=1.5, label=f"Mean: {df_all_bp[col].mean():.1f}")
    for val, lc, lbl in lines:
        axes[i].axvline(val, color=lc, linestyle=':', lw=1.5, label=lbl)
    axes[i].set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(f'{col} (mmHg)'); axes[i].set_ylabel('Densitas')
    axes[i].legend(loc='upper right'); axes[i].grid(True, linestyle='--', alpha=0.5)
plt.suptitle('Distribusi Ground Truth Blood Pressure (MIMIC-II)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Boxplot per part
plt.figure(figsize=(15, 6))
df_melted = df_all_bp.melt(id_vars=['Part'], value_vars=['SBP', 'DBP', 'MAP'], var_name='Target BP', value_name='mmHg')
sns.boxplot(data=df_melted, x='Part', y='mmHg', hue='Target BP', palette=['#1f77b4', '#2ca02c', '#ff7f0e'])
plt.title('Distribusi SBP, DBP, MAP per File Part Dataset', fontsize=13, fontweight='bold')
plt.xlabel('File Dataset'); plt.ylabel('Tekanan Darah (mmHg)')
plt.xticks(rotation=30, ha='right')
plt.grid(True, linestyle='--', alpha=0.4); plt.legend(title='Metrik BP', loc='upper right')
plt.tight_layout(); plt.show()

---
## 4. Visualisasi Sinyal Multi-Channel per Part

In [ ]:
num_parts = len(loaded_parts)
fig, axes = plt.subplots(num_parts, 1, figsize=(16, 3.2 * num_parts), sharex=False)
if num_parts == 1:
    axes = [axes]

for i, (part_name, cell_array) in enumerate(loaded_parts.items()):
    rec = None
    for r_idx in range(min(get_num_records(cell_array), 20)):
        try:
            candidate = extract_record_matrix(cell_array, r_idx)
        except Exception:
            continue
        if candidate is not None and candidate.ndim == 2 and candidate.shape[0] >= 3 and candidate.shape[1] >= EPOCH_LEN:
            rec = candidate[:, :EPOCH_LEN]
            break
    if rec is None:
        continue

    t = np.arange(EPOCH_LEN) / FS
    ax = axes[i]
    ax_abp = ax.twinx()
    l1 = ax.plot(t, rec[2,:], color='#1f77b4', lw=1.2, label='ECG Lead II')
    l2 = ax.plot(t, rec[0,:] * (np.std(rec[2,:])/(np.std(rec[0,:])+1e-6)) + np.mean(rec[2,:]),
                 color='#2ca02c', alpha=0.5, lw=1.0, linestyle='--', label='PPG (scaled)')
    l3 = ax_abp.plot(t, rec[1,:], color='#d62728', lw=1.3, alpha=0.85, label='ABP (mmHg)')
    ax.set_ylabel('ECG/PPG (a.u.)', color='#1f77b4', fontweight='bold')
    ax_abp.set_ylabel('ABP (mmHg)', color='#d62728', fontweight='bold')
    ax.set_title(f'Sampel Multi-Kanal (10s): {part_name}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Waktu (detik)'); ax.grid(True, linestyle='--', alpha=0.4)
    lines = l1 + l2 + l3
    ax.legend(lines, [l.get_label() for l in lines], loc='upper right', fontsize=9)

plt.suptitle('Visualisasi Sinyal Multi-Kanal per Part', fontsize=13, fontweight='bold', y=1.005)
plt.tight_layout(); plt.show()

---
## 5. Demonstrasi Preprocessing Step-by-Step (1 Epoch Contoh)

```
Sinyal Mentah → Notch 50Hz (Q=30) → Bandpass 0.5–40Hz → MinMax [0,1] → Sinyal Bersih
```

In [ ]:
# Ambil 1 epoch contoh
sample_cell = loaded_parts[list(loaded_parts.keys())[0]]
rec_sample = extract_record_matrix(sample_cell, 0)
epoch_ecg_raw = rec_sample[2, :EPOCH_LEN]
epoch_abp_gt  = rec_sample[1, :EPOCH_LEN]
epoch_time    = np.arange(EPOCH_LEN) / FS

# Step-by-step
step1_notch    = iir_notch_filter(epoch_ecg_raw, fs=FS, f0=50.0, q=30.0)
step2_bandpass = butterworth_bandpass_filter(step1_notch, fs=FS, lowcut=0.5, highcut=40.0, order=2)
step3_minmax   = minmax_scale(step2_bandpass)

# Verifikasi vs pipeline gabungan
from_pipeline = preprocess_ecg_pipeline(epoch_ecg_raw, fs=FS)
assert np.allclose(step3_minmax, from_pipeline, atol=1e-12), 'Pipeline tidak cocok!'
print('✅ Verifikasi: step-by-step 100% identik dengan pipeline gabungan')

# Visualisasi 4 panel
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(epoch_time, epoch_ecg_raw, color='#7f7f7f', lw=1.2)
axes[0].set_title('(a) ECG Mentah (Raw)', fontweight='bold')
axes[0].set_ylabel('Amplitudo (a.u.)')

axes[1].plot(epoch_time, step1_notch, color='#2ca02c', lw=1.2)
axes[1].set_title('(b) Setelah IIR Notch Filter 50 Hz (Q=30)', fontweight='bold')
axes[1].set_ylabel('Amplitudo (a.u.)')

axes[2].plot(epoch_time, step2_bandpass, color='#1f77b4', lw=1.2)
axes[2].set_title('(c) Setelah Butterworth Bandpass 0.5–40 Hz (Orde 2)', fontweight='bold')
axes[2].set_ylabel('Amplitudo (a.u.)')

axes[3].plot(epoch_time, step3_minmax, color='#9467bd', lw=1.5)
axes[3].axhline(0, color='k', ls=':', alpha=0.5); axes[3].axhline(1, color='k', ls=':', alpha=0.5)
axes[3].set_title('(d) Setelah Min-Max Normalization [0, 1] (Fan et al., 2021)', fontweight='bold')
axes[3].set_ylabel('Amplitudo [0,1]'); axes[3].set_ylim(-0.05, 1.05)
axes[3].set_xlabel('Waktu (detik)')

for ax in axes:
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Pipeline Preprocessing ECG — Step-by-Step', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

---
## 6. Batch Preprocessing Seluruh Dataset → Simpan .npz

Memproses **seluruh rekaman** dari setiap file part: epoching 10 detik → preprocessing → simpan `.npz`.

In [ ]:
# Bebaskan loaded_parts agar tidak double memory
del loaded_parts
gc.collect()

processed_files_summary = []

for mat_path in mat_files:
    part_name = os.path.basename(mat_path)
    part_stem = os.path.splitext(part_name)[0]
    out_path = os.path.join(PROCESSED_DIR, f'{part_stem}_preprocessed.npz')

    print('=' * 65)
    print(f'Memproses: {part_name} → {os.path.basename(out_path)}')

    cell_array = load_mat_file(mat_path)
    if cell_array is None:
        continue
    num_records = get_num_records(cell_array)

    X_part, y_part = [], []
    for r_idx in tqdm(range(num_records), desc=f'  {part_stem}'):
        try:
            rec = extract_record_matrix(cell_array, r_idx)
        except Exception:
            continue
        if rec is None or rec.ndim != 2 or rec.shape[0] < 3:
            continue

        raw_ecg = rec[2, :]
        raw_abp = rec[1, :]

        for start in range(0, len(raw_ecg) - EPOCH_LEN + 1, EPOCH_LEN):
            seg_ecg = raw_ecg[start:start+EPOCH_LEN]
            seg_abp = raw_abp[start:start+EPOCH_LEN]

            if np.std(seg_ecg) < 1e-4 or np.isnan(seg_ecg).any() or np.isnan(seg_abp).any():
                continue

            bp = compute_bp_metrics(seg_abp, fs=FS)
            if not bp['valid']:
                continue

            clean = preprocess_ecg_pipeline(seg_ecg, fs=FS)
            X_part.append(clean.astype(np.float32))
            y_part.append([bp['sbp'], bp['dbp'], bp['map']])

    X_part = np.array(X_part, dtype=np.float32)
    y_part = np.array(y_part, dtype=np.float32)

    if len(X_part) > 0:
        np.savez_compressed(out_path, X=X_part, y=y_part, fs=FS, part_name=part_name)
        size_mb = os.path.getsize(out_path) / 1024**2
        print(f'  ✅ {len(X_part):,} epoch → {out_path} ({size_mb:.1f} MB)')
        processed_files_summary.append({
            'Part': part_name, 'Epoch Valid': len(X_part),
            'Shape X': str(X_part.shape), 'Shape y': str(y_part.shape),
            'MB': round(size_mb, 2)
        })
    else:
        print(f'  ⚠️ Tidak ada epoch valid dari {part_name}')

    del cell_array, X_part, y_part
    gc.collect()

df_proc = pd.DataFrame(processed_files_summary)
print('\n--- Ringkasan Dataset Preprocessed ---')
display(df_proc)
if len(df_proc) > 0:
    print(f"\n📊 TOTAL: {df_proc['Epoch Valid'].sum():,} epoch | {df_proc['MB'].sum():.1f} MB")

---
# FASE II — TRAINING

---
## 7. Definisi Model & Loss (Inline)

### Arsitektur Multi-Task BiLSTM (Fan et al., 2021, Sensors)
```
ECG [batch, 1250, 1]
  │
  ▼
BiLSTM Layer 1 (hidden=256, bidirectional → 512) ─┐
  │ dropout=0.3                                    │
BiLSTM Layer 2 (hidden=256, bidirectional → 512)   │ Shared Encoder
  │                                                │
Terminal State [fwd_last ‖ bwd_first] → [batch, 512] ─┘
  │
  ├─→ SBP Head: FC(512→512)→ReLU→Drop→FC(512→256)→ReLU→Drop→FC(256→1)
  ├─→ DBP Head: FC(512→512)→ReLU→Drop→FC(512→256)→ReLU→Drop→FC(256→1)
  └─→ MAP Head: FC(512→512)→ReLU→Drop→FC(512→256)→ReLU→Drop→FC(256→1)
```

In [ ]:
class TaskSpecificHead(nn.Module):
    """FC Regression Head per target BP (Fan et al., 2021 Table 1)."""
    def __init__(self, in_features=512, h1=512, h2=256, dropout=0.2):
        super().__init__()
        layers = [nn.Linear(in_features, h1), nn.ReLU()]
        if dropout > 0: layers.append(nn.Dropout(dropout))
        layers.extend([nn.Linear(h1, h2), nn.ReLU()])
        if dropout > 0: layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(h2, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)


class ECGMultiTaskBiLSTM(nn.Module):
    """Multi-Task BiLSTM untuk estimasi simultan SBP, DBP, MAP."""
    def __init__(self, input_size=1, hidden_size=256, num_layers=2,
                 lstm_dropout=0.3, fc_dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True, bidirectional=True,
            dropout=lstm_dropout if num_layers > 1 else 0.0,
        )
        dim = hidden_size * 2
        self.sbp_head = TaskSpecificHead(dim, dim, dim//2, fc_dropout)
        self.dbp_head = TaskSpecificHead(dim, dim, dim//2, fc_dropout)
        self.map_head = TaskSpecificHead(dim, dim, dim//2, fc_dropout)

    def extract_features(self, x):
        if x.dim() == 2: x = x.unsqueeze(-1)
        out, _ = self.lstm(x)
        fwd_last  = out[:, -1, :self.hidden_size]
        bwd_first = out[:,  0, self.hidden_size:]
        return torch.cat([fwd_last, bwd_first], dim=-1)

    def forward(self, x):
        feat = self.extract_features(x)
        return self.sbp_head(feat), self.dbp_head(feat), self.map_head(feat)

    def predict_all(self, x):
        s, d, m = self.forward(x)
        return torch.cat([s, d, m], dim=-1)


class AdaptiveMultiTaskLoss(nn.Module):
    """Adaptive Weight Learning Loss (Fan et al., 2021 Algorithm 1)."""
    def __init__(self):
        super().__init__()
        self.register_buffer('weights', torch.tensor([1/3, 1/3, 1/3], dtype=torch.float32))
        self.prev_val_mean = None
        self.prev_val_std = None

    def forward(self, sbp_pred, dbp_pred, map_pred, targets):
        s_t, d_t, m_t = targets[:,0:1], targets[:,1:2], targets[:,2:3]
        sbp_pred = sbp_pred.view_as(s_t)
        dbp_pred = dbp_pred.view_as(d_t)
        map_pred = map_pred.view_as(m_t)
        ls = F.l1_loss(sbp_pred, s_t); ld = F.l1_loss(dbp_pred, d_t); lm = F.l1_loss(map_pred, m_t)
        total = self.weights[0]*ls + self.weights[1]*ld + self.weights[2]*lm
        return total, {
            'loss_total': total.item(),
            'mae_sbp': F.l1_loss(sbp_pred, s_t).item(),
            'mae_dbp': F.l1_loss(dbp_pred, d_t).item(),
            'mae_map': F.l1_loss(map_pred, m_t).item(),
        }

    def update_weights(self, sbp_l, dbp_l, map_l):
        eps = 1e-8
        cm = np.array([np.mean(sbp_l), np.mean(dbp_l), np.mean(map_l)])
        cs = np.array([np.std(sbp_l), np.std(dbp_l), np.std(map_l)])
        if self.prev_val_mean is not None:
            tm = np.abs(cm - self.prev_val_mean) / (cm + eps)
            ts = np.abs(cs - self.prev_val_std) / (cs + eps)
            raw = tm * ts * (cm + cs)
            s = np.sum(raw)
            nw = raw / s if s > eps else np.array([1/3,1/3,1/3])
            self.weights.copy_(torch.tensor(nw, dtype=torch.float32, device=self.weights.device))
        self.prev_val_mean = cm; self.prev_val_std = cs


# Info model
_m = ECGMultiTaskBiLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, LSTM_DROPOUT, FC_DROPOUT)
print(f'✅ ECGMultiTaskBiLSTM: {sum(p.numel() for p in _m.parameters()):,} parameter')
del _m

---
## 8. Muat Dataset Preprocessed & Buat DataLoader

In [ ]:
class ECGBPDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float().unsqueeze(-1)  # [N, 1250, 1]
        self.y = torch.from_numpy(y).float()                 # [N, 3]
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# Muat semua .npz preprocessed
npz_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, '*_preprocessed.npz')))
print(f'Memuat {len(npz_files)} file preprocessed...')

all_X, all_y = [], []
for fp in npz_files:
    d = np.load(fp)
    all_X.append(d['X']); all_y.append(d['y'])
    print(f'  • {os.path.basename(fp)}: {d["X"].shape[0]:,} epoch')

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)
del all_X, all_y; gc.collect()

print(f'\nTotal: X={X_all.shape}, y={y_all.shape} ({X_all.nbytes/1024**2:.1f} MB)')
for i, name in enumerate(['SBP', 'DBP', 'MAP']):
    v = y_all[:, i]
    print(f'  {name}: {v.mean():.1f} ± {v.std():.1f} | [{v.min():.1f}, {v.max():.1f}] mmHg')

# Buat dataset & split
dataset = ECGBPDataset(X_all, y_all)
total_len = len(dataset)
train_len = int(TRAIN_RATIO * total_len)
val_len = int(VAL_RATIO * total_len)
test_len = total_len - train_len - val_len

train_set, val_set, test_set = random_split(
    dataset, [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'\n📊 Split (Seed={SEED}): Train={len(train_set):,} | Val={len(val_set):,} | Test={len(test_set):,}')
print(f'   Steps/epoch: {len(train_loader)} | Batch: {BATCH_SIZE}')

---
## 9. Training Loop

In [ ]:
def train_one_epoch(model, criterion, optimizer, loader, device):
    model.train()
    tot, ms, md, mm, n = 0, 0, 0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        s, d, m = model(xb)
        loss, met = criterion(s, d, m, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()
        tot += met['loss_total']; ms += met['mae_sbp']; md += met['mae_dbp']; mm += met['mae_map']; n += 1
    return {'loss': tot/max(1,n), 'mae_sbp': ms/max(1,n), 'mae_dbp': md/max(1,n), 'mae_map': mm/max(1,n)}

@torch.no_grad()
def evaluate(model, criterion, loader, device, update_w=False):
    model.eval()
    tot, n = 0, 0
    sl, dl, ml = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        s, d, m = model(xb)
        loss, met = criterion(s, d, m, yb)
        tot += met['loss_total']; sl.append(met['mae_sbp']); dl.append(met['mae_dbp']); ml.append(met['mae_map']); n += 1
    if update_w: criterion.update_weights(sl, dl, ml)
    return {'loss': tot/max(1,n), 'mae_sbp': float(np.mean(sl)), 'mae_dbp': float(np.mean(dl)), 'mae_map': float(np.mean(ml))}

print('✅ Fungsi training & evaluasi siap.')

In [ ]:
# Inisialisasi model, loss, optimizer, scheduler sesuai Fan et al. (2021)
model = ECGMultiTaskBiLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, LSTM_DROPOUT, FC_DROPOUT).to(DEVICE)
criterion = AdaptiveMultiTaskLoss().to(DEVICE)
# Optimizer: Momentum-SGD (Fan et al., 2021) atau AdamW
if OPTIMIZER_TYPE == 'SGD':
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )
    print(f'✅ Optimizer: Momentum-SGD (lr={LEARNING_RATE}, momentum={MOMENTUM}, weight_decay={WEIGHT_DECAY}) [Fan et al., 2021]')
else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    print(f'✅ Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})')
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE)

best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
history = {'train_loss':[], 'val_loss':[], 'val_mae_sbp':[], 'val_mae_dbp':[], 'val_mae_map':[], 'lr':[], 'weights':[]}
best_val_mae = float('inf')

print('=' * 80)
print('  PELATIHAN: MULTI-TASK BiLSTM (ECG → SBP/DBP/MAP)')
print('  Referensi: Fan et al. (2021, Sensors)')
print('=' * 80)
print(f'  Device: {DEVICE} | Train: {len(train_set):,} | Epochs: {NUM_EPOCHS} | Opt: {OPTIMIZER_TYPE} | LR: {LEARNING_RATE}')
print('=' * 80)

t_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr = train_one_epoch(model, criterion, optimizer, train_loader, DEVICE)
    vr = evaluate(model, criterion, val_loader, DEVICE, update_w=True)
    scheduler.step(vr['loss'])
    lr_now = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0
    avg_mae = (vr['mae_sbp'] + vr['mae_dbp'] + vr['mae_map']) / 3.0

    history['train_loss'].append(tr['loss']); history['val_loss'].append(vr['loss'])
    history['val_mae_sbp'].append(vr['mae_sbp']); history['val_mae_dbp'].append(vr['mae_dbp'])
    history['val_mae_map'].append(vr['mae_map']); history['lr'].append(lr_now)
    history['weights'].append(criterion.weights.tolist())

    is_best = avg_mae < best_val_mae
    if is_best:
        best_val_mae = avg_mae
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_val_mae': best_val_mae, 'history': history}, best_model_path)
        mark = '★ BEST'
    else:
        mark = ''

    w = criterion.weights.tolist()
    print(f'Epoch [{epoch:02d}/{NUM_EPOCHS}] ({elapsed:.1f}s) | '
          f'TrLoss:{tr["loss"]:.4f} | VlLoss:{vr["loss"]:.4f} | '
          f'SBP:{vr["mae_sbp"]:.2f} DBP:{vr["mae_dbp"]:.2f} MAP:{vr["mae_map"]:.2f} mmHg {mark} | '
          f'θ=[{w[0]:.2f},{w[1]:.2f},{w[2]:.2f}] LR={lr_now:.1e}')

total_time = time.time() - t_start
print('\n' + '=' * 80)
print(f'Pelatihan selesai: {total_time/60:.1f} menit ({total_time/3600:.2f} jam)')
print(f'Model terbaik (Avg Val MAE: {best_val_mae:.2f} mmHg) → {best_model_path}')
print('=' * 80)

---
## 10. Visualisasi Kurva Training

In [ ]:
ep = np.arange(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].plot(ep, history['train_loss'], 'o-', label='Train', color='#1f77b4', lw=2)
axes[0,0].plot(ep, history['val_loss'], 's--', label='Val', color='#ff7f0e', lw=2)
axes[0,0].set_title('Loss (Train vs Val)', fontweight='bold'); axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Weighted MAE')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(ep, history['val_mae_sbp'], 'o-', label='SBP', color='#e74c3c', lw=2)
axes[0,1].plot(ep, history['val_mae_dbp'], 's-', label='DBP', color='#3498db', lw=2)
axes[0,1].plot(ep, history['val_mae_map'], '^-', label='MAP', color='#2ecc71', lw=2)
axes[0,1].axhline(5.0, color='gray', ls=':', alpha=0.7, label='Target AAMI ≤5')
axes[0,1].set_title('Val MAE per Target', fontweight='bold'); axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('MAE (mmHg)')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(ep, history['lr'], 'D-', color='#9b59b6', lw=2)
axes[1,0].set_title('Learning Rate', fontweight='bold'); axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('LR')
axes[1,0].set_yscale('log'); axes[1,0].grid(True, alpha=0.3)

wa = np.array(history['weights'])
axes[1,1].plot(ep, wa[:,0], 'o-', label='θ_SBP', color='#e74c3c', lw=2)
axes[1,1].plot(ep, wa[:,1], 's-', label='θ_DBP', color='#3498db', lw=2)
axes[1,1].plot(ep, wa[:,2], '^-', label='θ_MAP', color='#2ecc71', lw=2)
axes[1,1].axhline(1/3, color='gray', ls=':', alpha=0.5, label='Init (1/3)')
axes[1,1].set_title('Adaptive Weights (θ)', fontweight='bold'); axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('θ')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Monitoring Pelatihan Multi-Task BiLSTM', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

---
# FASE III — EVALUASI

---
## 11. Evaluasi Akhir pada Test Set

In [ ]:
ckpt = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Model terbaik dimuat dari epoch {ckpt["epoch"]} (Avg Val MAE: {ckpt["best_val_mae"]:.2f} mmHg)')

all_preds, all_trues = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_loader, desc='Evaluasi Test Set'):
        preds = model.predict_all(xb.to(DEVICE))
        all_preds.append(preds.cpu().numpy())
        all_trues.append(yb.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_trues = np.concatenate(all_trues, axis=0)
errors = all_preds - all_trues
abs_errors = np.abs(errors)

target_names = ['SBP', 'DBP', 'MAP']
print('\n' + '=' * 70)
print('  HASIL EVALUASI AKHIR — TEST SET (DATA UNSEEN)')
print('  Standar: AAMI SP10 & BHS Protocol')
print('=' * 70)

results = {}
for i, name in enumerate(target_names):
    mae = np.mean(abs_errors[:,i]); std = np.std(errors[:,i])
    me = np.mean(errors[:,i]); rmse = np.sqrt(np.mean(errors[:,i]**2))
    results[name] = {'MAE': mae, 'STD': std, 'ME': me, 'RMSE': rmse}

    p5 = np.mean(abs_errors[:,i]<=5)*100; p10 = np.mean(abs_errors[:,i]<=10)*100; p15 = np.mean(abs_errors[:,i]<=15)*100
    if p5>=60 and p10>=85 and p15>=95: bhs = 'A'
    elif p5>=50 and p10>=75 and p15>=90: bhs = 'B'
    elif p5>=40 and p10>=65 and p15>=85: bhs = 'C'
    else: bhs = 'D'
    aami = '✅ LULUS' if mae<=5 and std<=8 else '❌ GAGAL'

    print(f'\n  {name}:')
    print(f'    MAE: {mae:.2f} mmHg | ME: {me:+.2f} ± {std:.2f} | RMSE: {rmse:.2f}')
    print(f'    BHS: ≤5={p5:.1f}% ≤10={p10:.1f}% ≤15={p15:.1f}% → Grade {bhs}')
    print(f'    AAMI SP10: {aami}')

avg_mae = np.mean([r['MAE'] for r in results.values()])
print(f'\n  Rata-rata MAE Keseluruhan: {avg_mae:.2f} mmHg')
print('=' * 70)

---
## 12. Scatter Plot: Prediksi vs Ground Truth

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']
for i, (name, c) in enumerate(zip(target_names, colors)):
    t, p = all_trues[:,i], all_preds[:,i]
    axes[i].scatter(t, p, alpha=0.15, s=8, color=c, edgecolors='none')
    lims = [min(t.min(), p.min())-5, max(t.max(), p.max())+5]
    axes[i].plot(lims, lims, 'k--', lw=1.5, alpha=0.7, label='Ideal (y=x)')
    axes[i].fill_between(lims, [l-5 for l in lims], [l+5 for l in lims], alpha=0.1, color='green', label='±5 mmHg')
    axes[i].set_title(f'{name} — MAE: {results[name]["MAE"]:.2f} mmHg', fontweight='bold', fontsize=12)
    axes[i].set_xlabel(f'{name} Ground Truth (mmHg)'); axes[i].set_ylabel(f'{name} Prediksi (mmHg)')
    axes[i].set_xlim(lims); axes[i].set_ylim(lims); axes[i].set_aspect('equal')
    axes[i].legend(loc='upper left', fontsize=9); axes[i].grid(True, alpha=0.3)
plt.suptitle(f'Prediksi vs Ground Truth — Test Set ({len(all_trues):,} sampel)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'scatter_pred_vs_true.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 13. Bland-Altman Agreement Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, c) in enumerate(zip(target_names, colors)):
    t, p = all_trues[:,i], all_preds[:,i]
    mean_v = (t + p) / 2; diff_v = p - t
    md, sd = np.mean(diff_v), np.std(diff_v)
    axes[i].scatter(mean_v, diff_v, alpha=0.1, s=6, color=c, edgecolors='none')
    axes[i].axhline(md, color='black', ls='-', lw=1.5, label=f'Bias: {md:+.2f}')
    axes[i].axhline(md+1.96*sd, color='red', ls='--', lw=1, label=f'+1.96SD: {md+1.96*sd:+.2f}')
    axes[i].axhline(md-1.96*sd, color='red', ls='--', lw=1, label=f'−1.96SD: {md-1.96*sd:+.2f}')
    axes[i].axhline(0, color='gray', ls=':', alpha=0.5)
    axes[i].set_title(f'Bland-Altman: {name}', fontweight='bold', fontsize=12)
    axes[i].set_xlabel(f'Mean (mmHg)'); axes[i].set_ylabel(f'Diff (Pred−True) (mmHg)')
    axes[i].legend(fontsize=8, loc='upper right'); axes[i].grid(True, alpha=0.3)
plt.suptitle('Bland-Altman Agreement — Test Set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'bland_altman.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 14. Simpan Semua Artefak

In [ ]:
# Training history
with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

# Evaluation results
eval_results = {
    'test_samples': len(all_trues),
    'results': {n: {k: float(v) for k, v in r.items()} for n, r in results.items()},
    'avg_mae': float(avg_mae),
    'best_epoch': ckpt['epoch'],
    'hyperparameters': {
        'batch_size': BATCH_SIZE, 'num_epochs': NUM_EPOCHS, 'lr': LEARNING_RATE,
        'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS,
        'lstm_dropout': LSTM_DROPOUT, 'fc_dropout': FC_DROPOUT,
        'grad_clip': GRAD_CLIP,
        'optimizer': f'Momentum-SGD (momentum={MOMENTUM})' if OPTIMIZER_TYPE == 'SGD' else 'AdamW',
        'weight_decay': WEIGHT_DECAY,
        'scheduler': f'ReduceLROnPlateau (factor={SCHEDULER_FACTOR}, patience={SCHEDULER_PATIENCE})',
        'preprocessing': f'Notch {NOTCH_FREQ}Hz (Q={NOTCH_Q}) → BPF {BP_LOWCUT}-{BP_HIGHCUT}Hz → MinMax [0,1]',
    }
}
with open(os.path.join(OUTPUT_DIR, 'evaluation_results.json'), 'w') as f:
    json.dump(eval_results, f, indent=2)

# Final model (lengkap untuk reproduksi)
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {'input_size': INPUT_SIZE, 'hidden_size': HIDDEN_SIZE, 'num_layers': NUM_LAYERS,
                     'lstm_dropout': LSTM_DROPOUT, 'fc_dropout': FC_DROPOUT},
    'preprocessing_config': {'fs': FS, 'seq_len': EPOCH_LEN, 'notch_freq': NOTCH_FREQ, 'notch_q': NOTCH_Q,
                             'bp_lowcut': BP_LOWCUT, 'bp_highcut': BP_HIGHCUT, 'bp_order': BP_ORDER,
                             'normalization': 'minmax [0,1]'},
    'evaluation': eval_results,
}, os.path.join(OUTPUT_DIR, 'final_model_full.pt'))

print('\n✅ Semua artefak tersimpan di /kaggle/working/')
print('\n📁 Daftar file output:')
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*'))):
    if os.path.isfile(f):
        print(f'  • {os.path.basename(f)} ({os.path.getsize(f)/1024**2:.2f} MB)')
    elif os.path.isdir(f):
        n = len(os.listdir(f))
        print(f'  📂 {os.path.basename(f)}/ ({n} files)')

print('\n   Unduh dari panel Output di sebelah kanan Kaggle notebook.')

---
## Ringkasan File Output

| File | Deskripsi |
|---|---|
| `processed/*.npz` | Dataset ECG preprocessed per part (X=[N,1250], y=[N,3]) |
| `checkpoints/best_model.pt` | Checkpoint model terbaik (berdasarkan Val MAE) |
| `final_model_full.pt` | Model + config + evaluasi (untuk deployment) |
| `training_history.json` | Riwayat loss, MAE, LR, bobot adaptif per epoch |
| `evaluation_results.json` | Metrik evaluasi akhir (MAE, RMSE, BHS, AAMI) |
| `training_curves.png` | Kurva konvergensi training |
| `scatter_pred_vs_true.png` | Scatter plot prediksi vs ground truth |
| `bland_altman.png` | Bland-Altman agreement analysis |